In [1]:
import os

SYSTEM_CA = "/etc/ssl/certs/ca-certificates.crt"

os.environ["REQUESTS_CA_BUNDLE"] = SYSTEM_CA
os.environ["SSL_CERT_FILE"] = SYSTEM_CA
os.environ["CURL_CA_BUNDLE"] = SYSTEM_CA

print("Using certificate bundle:", SYSTEM_CA)



from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

Using certificate bundle: /etc/ssl/certs/ca-certificates.crt


/home/nineleaps/jupyter_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

# --------------------------------------------------
# Task 1: Knowledge Base and Embeddings
# --------------------------------------------------

knowledge_base = [
    "To reset your password, click on the Forgot Password link on the login page.",
    "You can update your account email address from the account settings page.",
    "If you cannot log in, make sure that your username and password are correct.",
    "You can view your billing history and invoices from the billing section.",
    "To update your payment method, go to billing settings and select Payment Methods.",
    "If your account is locked, contact customer support to unlock your account.",
    "You can change your profile information from the account management section.",
    "A password reset link will be sent to your registered email address.",
    "If you were charged twice, contact the billing support team for assistance.",
    "You can cancel your subscription from the subscription management page.",
    "Two-factor authentication can be enabled from your account security settings.",
    "If you forgot your username, use the account recovery option on the login page."
]


# Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Generate embeddings
embeddings = model.encode(knowledge_base)

# Convert to float32 because FAISS expects float32
embeddings = np.array(embeddings).astype("float32")

print("Embedding shape:", embeddings.shape)


# --------------------------------------------------
# Task 2: Build FAISS Index
# --------------------------------------------------

# Normalize embeddings
faiss.normalize_L2(embeddings)

# Create FAISS index
dimension = 384
index = faiss.IndexFlatL2(dimension)

# Add embeddings to index
index.add(embeddings)

print("Total vectors stored:", index.ntotal)


# --------------------------------------------------
# Task 3: Semantic Search
# --------------------------------------------------

def semantic_search(query, k=3):

    # Convert query into embedding
    query_embedding = model.encode([query])

    # Convert to float32
    query_embedding = np.array(query_embedding).astype("float32")

    # Normalize query embedding
    faiss.normalize_L2(query_embedding)

    # Search FAISS index
    distances, indices = index.search(query_embedding, k)

    print("\nQuery:", query)
    print("-" * 80)
    print(f"{'Rank':<8}{'Score':<15}Matched Sentence")
    print("-" * 80)

    for rank, (distance, idx) in enumerate(
        zip(distances[0], indices[0]), start=1
    ):
        print(
            f"{rank:<8}{distance:<15.4f}{knowledge_base[idx]}"
        )


# Test Query 1
semantic_search("How can I change my password?")

# Test Query 2
semantic_search("I was charged twice for my payment")

# Test Query 3
semantic_search("I cannot access my account")


# --------------------------------------------------
# Task 4: Interactive CLI
# --------------------------------------------------

print("\n" + "=" * 80)
print("Interactive Semantic Search")
print("Type 'exit' to quit")
print("=" * 80)

while True:

    query = input("\nEnter your query: ")

    if query.lower() == "exit":
        print("Exiting semantic search...")
        break

    semantic_search(query)

Loading weights: 100%|██████████████████████| 103/103 [00:00<00:00, 839.36it/s]


Embedding shape: (12, 384)
Total vectors stored: 12

Query: How can I change my password?
--------------------------------------------------------------------------------
Rank    Score          Matched Sentence
--------------------------------------------------------------------------------
1       0.6983         To reset your password, click on the Forgot Password link on the login page.
2       0.8743         A password reset link will be sent to your registered email address.
3       0.9429         If your account is locked, contact customer support to unlock your account.

Query: I was charged twice for my payment
--------------------------------------------------------------------------------
Rank    Score          Matched Sentence
--------------------------------------------------------------------------------
1       0.4037         If you were charged twice, contact the billing support team for assistance.
2       1.1814         To update your payment method, go to billing setti


Enter your query:  I forgot my password. How can I reset it?



Query: I forgot my password. How can I reset it?
--------------------------------------------------------------------------------
Rank    Score          Matched Sentence
--------------------------------------------------------------------------------
1       0.3569         To reset your password, click on the Forgot Password link on the login page.
2       0.5850         A password reset link will be sent to your registered email address.
3       0.6684         If you forgot your username, use the account recovery option on the login page.



Enter your query:  How can I update my account information?



Query: How can I update my account information?
--------------------------------------------------------------------------------
Rank    Score          Matched Sentence
--------------------------------------------------------------------------------
1       0.6710         You can change your profile information from the account management section.
2       0.7513         You can update your account email address from the account settings page.
3       1.0315         To update your payment method, go to billing settings and select Payment Methods.



Enter your query:  exit


Exiting semantic search...
